# 🧪 Experiment Benchmark: Pure BERT vs. Edit Distance + BERT (Fair 5-Fold CV)

สมุดโน้ตเล่มนี้ได้รับการปรับปรุงเพื่อ **แก้ไขปัญหา Data Leakage** และทำการวัดผลเชิงวิทยาศาสตร์อย่างถูกต้องเที่ยงตรงด้วย **5-Fold Stratified Cross-Validation**

---

### 🔬 5 วิธีที่นำมาทดสอบเปรียบเทียบ (Evaluation Methods):
1. **Method 1: Pure BERT (`intfloat/multilingual-e5-small`)**
   - ส่งข้อความดั้งเดิม (`raw_query`) ตรงเข้า BERT โดย **ไม่ผ่าน** Edit Distance หรือ Rule ใดๆ
2. **Method 2: Edit Distance + BERT**
   - ทำการแก้คำผิดด้วย Domain Vocab Edit Distance (Tier 0) ก่อนส่งเข้า BERT
3. **Method 3: Tier 1 Priority Rules + Pure BERT**
   - ใช้ Priority Rules กรองคีย์เวิร์ดเด่นก่อน (งบ/ราคา/สเปค) แล้วเคสเปิดส่งเข้า Pure BERT
4. **Method 4a: 4-Tier Pipeline (No Few-Shot)**
   - สถาปัตยกรรม (Tier 0 Spell Fix + Tier 1 Rules + Tier 3 BERT Fallback) โดยข้าม ChromaDB Few-Shot
5. **Method 4b: Full 4-Tier Pipeline (With Leak-Free 5-Fold CV Few-Shot)**
   - สถาปัตยกรรม 4-Tier ครบชุด โดยสร้าง ChromaDB Few-Shot Index จาก Train Fold (80%) เท่านั้น และทดสอบบน Test Fold (20%) เพื่อการวัดผลที่ปราศจาก Data Leakage 100%

---
📌 **คำแนะนำ:** สามารถเปิดสมุดโน้ตนี้ใน Jupyter / VS Code แล้วสั่ง **Run All Cells** เพื่อรันการประมวลผล 5-Fold CV และดูตารางสรุปผลได้ทันทีครับ

## 🛠️ Step 1: โหลดไลบรารี ชุดข้อมูล Ground Truth และ Model Warmup

In [1]:
import sys
import os
import json
import time
import re
from typing import Dict, List, Any, Tuple
import pandas as pd
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold

# ตั้งค่า Encoding Safeguard สำหรับ Windows CLI (Guard สำหรับ Jupyter OutStream)
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

# ระบุ Path ไปยัง Root Project
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from app.services.intent_service import IntentService

# 1. โหลด Ground Truth Dataset (136 ตัวอย่าง)
gt_path = os.path.join(project_root, "app", "data", "nlp_ground_truth.json")
with open(gt_path, "r", encoding="utf-8") as f:
    ground_truth_data = json.load(f)

print(f"✅ โหลด nlp_ground_truth.json สำเร็จ! จำนวนทั้งหมด {len(ground_truth_data)} ตัวอย่าง")
typo_count = sum(1 for item in ground_truth_data if item.get('has_typos', False))
clean_count = len(ground_truth_data) - typo_count
print(f"   - ข้อความที่มีคำพิมพ์ผิด (Typo Queries): {typo_count} ประโยค")
print(f"   - ข้อความสะกดถูกต้อง (Clean Queries): {clean_count} ประโยค")

# 2. โหลดโมเดล BERT (intfloat/multilingual-e5-small)
print("⏳ กำลังโหลดโมเดล BERT: intfloat/multilingual-e5-small...")
bert_model = SentenceTransformer('intfloat/multilingual-e5-small')
print("✅ โหลดโมเดล BERT สำเร็จ!")

# 3. Model Warmup (ยิงประโยคดัมมี่เพื่อเคลียร์ Overhead และ First-call Latency)
dummy_warmup = bert_model.encode(["query: Warmup inference sentence"], convert_to_tensor=True)
print("🔥 Model Warmup สำเร็จพร้อมสำหรับวัด Latency เที่ยงตรง!")

# 4. เริ่มต้น IntentService ตัวแม่
intent_service = IntentService(data_dir=os.path.join(project_root, "app", "data"))
print("✅ เริ่มต้น IntentService สำเร็จ!")


✅ โหลด nlp_ground_truth.json สำเร็จ! จำนวนทั้งหมด 136 ตัวอย่าง
   - ข้อความที่มีคำพิมพ์ผิด (Typo Queries): 11 ประโยค
   - ข้อความสะกดถูกต้อง (Clean Queries): 125 ประโยค
⏳ กำลังโหลดโมเดล BERT: intfloat/multilingual-e5-small...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ โหลดโมเดล BERT สำเร็จ!
🔥 Model Warmup สำเร็จพร้อมสำหรับวัด Latency เที่ยงตรง!


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ เริ่มต้น IntentService สำเร็จ!


## 📐 Step 2: นิยาม Intent Passages & Reference Embeddings

In [2]:
INTENT_PASSAGES = {
    "product_search": "passage: ซื้อเสื้อ หาเสื้อ ขอดูเสื้อ สั่งซื้อเสื้อผ้า เสื้อยืด กางเกง เสื้อโปโล เสื้อเชิ้ต ราคาสินค้า สี ไซส์ ทรงเสื้อ มีงบ มีราคา ไม่เกิน",
    "size_recommendation": "passage: สอบถามไซส์ แนะนำไซส์เสื้อ ขนาดเสื้อ รอบอก สัดส่วนความสูงและน้ำหนัก ไซส์ไหนดี ใส่ไซส์อะไร เหมาะกับไซส์อะไร",
    "fabric_comparison": "passage: สอบถามเนื้อผ้า เปรียบเทียบคุณสมบัติผ้า ผ้าต่างกันยังไง ซักแล้วยับไหม ผ้านุ่ม ระบายอากาศ ดีกว่ายังไง คุณสมบัติของผ้า",
    "promotion_discount": "passage: โปรโมชัน ดีลพิเศษ ประจำวัน ประจำเดือน แฟลชเซล ส่วนลด คูปอง ลดราคา วันนี้ เดือนนี้ ส่งฟรี"
}
intent_classes = list(INTENT_PASSAGES.keys())
passage_embeddings = bert_model.encode(list(INTENT_PASSAGES.values()), convert_to_tensor=True)
print(f"✅ คำนวณ Passage Embeddings สำหรับ {len(intent_classes)} Intent เรียบร้อยแล้ว")


✅ คำนวณ Passage Embeddings สำหรับ 4 Intent เรียบร้อยแล้ว


## 🧪 Step 3: นิยามฟังก์ชันทำนายผลทั้ง 5 สถาปัตยกรรม

In [3]:
# --- Method 1: Pure BERT (ส่ง raw query ตรงเข้า BERT) ---
def predict_method_1_pure_bert(raw_query: str) -> Tuple[str, float]:
    start_t = time.perf_counter()
    query_text = f"query: {raw_query}"
    query_embedding = bert_model.encode(query_text, convert_to_tensor=True)
    scores = util.cos_sim(query_embedding, passage_embeddings)[0]
    best_idx = int(scores.argmax())
    latency = (time.perf_counter() - start_t) * 1000.0
    return intent_classes[best_idx], latency

# --- Method 2: Edit Distance + BERT (ผ่าน Tier 0 แก้คำผิดก่อนเข้า BERT) ---
def predict_method_2_editdistance_bert(raw_query: str) -> Tuple[str, float]:
    start_t = time.perf_counter()
    corrected_query, _ = intent_service.correct_spelling(raw_query)
    query_text = f"query: {corrected_query}"
    query_embedding = bert_model.encode(query_text, convert_to_tensor=True)
    scores = util.cos_sim(query_embedding, passage_embeddings)[0]
    best_idx = int(scores.argmax())
    latency = (time.perf_counter() - start_t) * 1000.0
    return intent_classes[best_idx], latency

# --- Method 3: Tier 1 Priority Rules + Pure BERT ---
def predict_method_3_rules_pure_bert(raw_query: str) -> Tuple[str, float]:
    start_t = time.perf_counter()
    raw_lower = raw_query.lower()
    
    if (re.search(r'(?:สูง|หนัก)\s*\d+', raw_lower) or re.search(r'(?:ไซส์|ขนาด)(?:อะไร|ไหน|เท่าไหร่|ดี)', raw_lower)) and not re.search(r'(?:ไม่เกิน|งบ|ราคา|บาท)', raw_lower):
        return "size_recommendation", (time.perf_counter() - start_t) * 1000.0
    if any(ft in raw_lower for ft in ["ต่างกันยังไง", "ต่างกับ", "ดีกว่ายังไง", "คุณสมบัติ", "ผ้านุ่ม"]) and not re.search(r'(?:ไม่เกิน|งบ)', raw_lower):
        return "fabric_comparison", (time.perf_counter() - start_t) * 1000.0
    if any(pt in raw_lower for pt in ["โปรโมชัน", "โปรโมชั่น", "ส่วนลด", "คูปอง", "แฟลชเซล"]) and not re.search(r'(?:ไม่เกิน|งบ)', raw_lower):
        return "promotion_discount", (time.perf_counter() - start_t) * 1000.0
    if any(pt in raw_lower for pt in ["ไม่เกิน", "งบ", "บาท", "ขอดู", "หาเสื้อ", "ราคาประมาณ"]):
        return "product_search", (time.perf_counter() - start_t) * 1000.0
        
    query_text = f"query: {raw_query}"
    query_embedding = bert_model.encode(query_text, convert_to_tensor=True)
    scores = util.cos_sim(query_embedding, passage_embeddings)[0]
    best_idx = int(scores.argmax())
    latency = (time.perf_counter() - start_t) * 1000.0
    return intent_classes[best_idx], latency

# --- Method 4a: 4-Tier Pipeline (No Few-Shot Lookup) ---
def predict_method_4a_no_fewshot(raw_query: str) -> Tuple[str, float]:
    start_t = time.perf_counter()
    corrected_query, _ = intent_service.correct_spelling(raw_query)
    raw_lower = raw_query.lower()
    
    # Tier 1 Priority Rules
    if (re.search(r'(?:สูง|หนัก)\s*\d+', raw_lower) or re.search(r'(?:ไซส์|ขนาด)(?:อะไร|ไหน|เท่าไหร่|ดี)', raw_lower)) and not re.search(r'(?:ไม่เกิน|งบ|ราคา|บาท)', raw_lower):
        return "size_recommendation", (time.perf_counter() - start_t) * 1000.0
    if any(ft in raw_lower for ft in ["ต่างกันยังไง", "ต่างกับ", "ดีกว่ายังไง", "คุณสมบัติ", "ผ้านุ่ม"]) and not re.search(r'(?:ไม่เกิน|งบ)', raw_lower):
        return "fabric_comparison", (time.perf_counter() - start_t) * 1000.0
    if any(pt in raw_lower for pt in ["โปรโมชัน", "โปรโมชั่น", "ส่วนลด", "คูปอง", "แฟลชเซล"]) and not re.search(r'(?:ไม่เกิน|งบ)', raw_lower):
        return "promotion_discount", (time.perf_counter() - start_t) * 1000.0
    if any(pt in raw_lower for pt in ["ไม่เกิน", "งบ", "บาท", "ขอดู", "หาเสื้อ", "ราคาประมาณ"]):
        return "product_search", (time.perf_counter() - start_t) * 1000.0
        
    # Tier 3 BERT Passage Match (Skip Tier 2.5 Few-Shot)
    query_text = f"query: {corrected_query}"
    query_embedding = bert_model.encode(query_text, convert_to_tensor=True)
    scores = util.cos_sim(query_embedding, passage_embeddings)[0]
    best_idx = int(scores.argmax())
    latency = (time.perf_counter() - start_t) * 1000.0
    return intent_classes[best_idx], latency

# --- Method 4b: Full 4-Tier Pipeline (With Leak-Free Few-Shot Collection) ---
def predict_method_4b_cv_fewshot(raw_query: str, cv_chroma_collection) -> Tuple[str, float]:
    start_t = time.perf_counter()
    corrected_query, _ = intent_service.correct_spelling(raw_query)
    raw_lower = raw_query.lower()
    
    # Tier 1 Priority Rules
    if (re.search(r'(?:สูง|หนัก)\s*\d+', raw_lower) or re.search(r'(?:ไซส์|ขนาด)(?:อะไร|ไหน|เท่าไหร่|ดี)', raw_lower)) and not re.search(r'(?:ไม่เกิน|งบ|ราคา|บาท)', raw_lower):
        return "size_recommendation", (time.perf_counter() - start_t) * 1000.0
    if any(ft in raw_lower for ft in ["ต่างกันยังไง", "ต่างกับ", "ดีกว่ายังไง", "คุณสมบัติ", "ผ้านุ่ม"]) and not re.search(r'(?:ไม่เกิน|งบ)', raw_lower):
        return "fabric_comparison", (time.perf_counter() - start_t) * 1000.0
    if any(pt in raw_lower for pt in ["โปรโมชัน", "โปรโมชั่น", "ส่วนลด", "คูปอง", "แฟลชเซล"]) and not re.search(r'(?:ไม่เกิน|งบ)', raw_lower):
        return "promotion_discount", (time.perf_counter() - start_t) * 1000.0
    if any(pt in raw_lower for pt in ["ไม่เกิน", "งบ", "บาท", "ขอดู", "หาเสื้อ", "ราคาประมาณ"]):
        return "product_search", (time.perf_counter() - start_t) * 1000.0
        
    # Tier 2.5 ChromaDB Few-Shot Search (Using CV Train Fold Collection)
    if cv_chroma_collection:
        query_emb = bert_model.encode(f"query: {raw_query}", convert_to_tensor=False).tolist()
        results = cv_chroma_collection.query(query_embeddings=[query_emb], n_results=1)
        if results and results.get("distances") and results["distances"][0]:
            dist = results["distances"][0][0]
            similarity = 1.0 - dist
            top_intent = results["metadatas"][0][0]["intent"]
            if similarity >= 0.70:
                return top_intent, (time.perf_counter() - start_t) * 1000.0
                
    # Tier 3 BERT Passage Match
    query_text = f"query: {corrected_query}"
    query_embedding = bert_model.encode(query_text, convert_to_tensor=True)
    scores = util.cos_sim(query_embedding, passage_embeddings)[0]
    best_idx = int(scores.argmax())
    latency = (time.perf_counter() - start_t) * 1000.0
    return intent_classes[best_idx], latency

print("✅ สร้างฟังก์ชันทำนายผลทั้ง 5 สถาปัตยกรรมเรียบร้อยแล้ว")


✅ สร้างฟังก์ชันทำนายผลทั้ง 5 สถาปัตยกรรมเรียบร้อยแล้ว


## 📊 Step 4: ประมวลผล 5-Fold Stratified Cross-Validation (Leak-Free Execution)

In [4]:
# เตรียมระบบ ChromaDB ในหน่วยความจำชั่วคราวสำหรับ Cross-Validation
chroma_ephemeral = chromadb.Client()

X = np.array([item["query"] for item in ground_truth_data])
y = np.array([item["expected_intent"] for item in ground_truth_data])
has_typo_arr = np.array([item.get("has_typos", False) for item in ground_truth_data])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

method_names = [
    "1. Pure BERT",
    "2. Edit Distance + BERT",
    "3. Rules + Pure BERT",
    "4a. 4-Tier Pipeline (No FewShot)",
    "4b. Full 4-Tier (Leak-Free 5-Fold CV)"
]

cv_results = {m: {"y_true": [], "y_pred": [], "latencies": [], "typo_correct": 0, "typo_total": 0, "clean_correct": 0, "clean_total": 0} for m in method_names}
detailed_logs = []

print("🚀 เริ่มการประมวลผล 5-Fold Stratified Cross-Validation (100% Data Leakage Protection)...")

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    print(f"   🔄 กำลังประมวลผล Fold {fold_idx + 1}/5 (Train: {len(train_idx)}, Test: {len(test_idx)})...")
    
    # สร้าง ChromaDB Collection พิเศษเฉพาะ Train Fold สำหรับ Method 4b
    coll_name = f"cv_fold_{fold_idx}"
    try:
        chroma_ephemeral.delete_collection(coll_name)
    except Exception:
        pass
    cv_collection = chroma_ephemeral.create_collection(coll_name)
    
    train_docs = [f"query: {X[i]}" for i in train_idx]
    train_embs = bert_model.encode(train_docs, convert_to_tensor=False).tolist()
    train_ids = [f"tr_{i}" for i in train_idx]
    train_metas = [{"intent": y[i], "query": X[i]} for i in train_idx]
    cv_collection.add(ids=train_ids, embeddings=train_embs, metadatas=train_metas, documents=[X[i] for i in train_idx])
    
    # ทดสอบประมวลผลใน Test Set
    for idx in test_idx:
        q = X[idx]
        expected = y[idx]
        has_typo = has_typo_arr[idx]
        
        # รันทั้ง 5 วิธี
        preds = {}
        preds["1. Pure BERT"] = predict_method_1_pure_bert(q)
        preds["2. Edit Distance + BERT"] = predict_method_2_editdistance_bert(q)
        preds["3. Rules + Pure BERT"] = predict_method_3_rules_pure_bert(q)
        preds["4a. 4-Tier Pipeline (No FewShot)"] = predict_method_4a_no_fewshot(q)
        preds["4b. Full 4-Tier (Leak-Free 5-Fold CV)"] = predict_method_4b_cv_fewshot(q, cv_collection)
        
        for m_name in method_names:
            pred_intent, lat = preds[m_name]
            is_corr = (pred_intent == expected)
            
            cv_results[m_name]["y_true"].append(expected)
            cv_results[m_name]["y_pred"].append(pred_intent)
            cv_results[m_name]["latencies"].append(lat)
            
            if has_typo:
                cv_results[m_name]["typo_total"] += 1
                if is_corr:
                    cv_results[m_name]["typo_correct"] += 1
            else:
                cv_results[m_name]["clean_total"] += 1
                if is_corr:
                    cv_results[m_name]["clean_correct"] += 1
                    
            detailed_logs.append({
                "Fold": fold_idx + 1,
                "Method": m_name,
                "Query": q,
                "Has_Typo": has_typo,
                "Expected": expected,
                "Predicted": pred_intent,
                "Is_Correct": is_corr,
                "Latency_ms": lat
            })

print("✅ สำเร็จ! ประมวลผล 5-Fold Cross Validation ปราศจาก Data Leakage 100%")


🚀 เริ่มการประมวลผล 5-Fold Stratified Cross-Validation (100% Data Leakage Protection)...
   🔄 กำลังประมวลผล Fold 1/5 (Train: 108, Test: 28)...
   🔄 กำลังประมวลผล Fold 2/5 (Train: 109, Test: 27)...
   🔄 กำลังประมวลผล Fold 3/5 (Train: 109, Test: 27)...
   🔄 กำลังประมวลผล Fold 4/5 (Train: 109, Test: 27)...
   🔄 กำลังประมวลผล Fold 5/5 (Train: 109, Test: 27)...
✅ สำเร็จ! ประมวลผล 5-Fold Cross Validation ปราศจาก Data Leakage 100%


## 🏆 Step 5: ตารางสรุปผลการวัดผลที่เที่ยงตรง (Final Fair Benchmark Results)

In [5]:
summary_data = []
for m_name in method_names:
    data = cv_results[m_name]
    acc = accuracy_score(data["y_true"], data["y_pred"]) * 100.0
    f1 = f1_score(data["y_true"], data["y_pred"], average="macro")
    avg_lat = np.mean(data["latencies"])
    p95_lat = np.percentile(data["latencies"], 95)
    
    typo_acc = (data["typo_correct"] / data["typo_total"] * 100.0) if data["typo_total"] > 0 else 0.0
    clean_acc = (data["clean_correct"] / data["clean_total"] * 100.0) if data["clean_total"] > 0 else 0.0
    
    summary_data.append({
        "Architecture Method": m_name,
        "Fair Accuracy (%)": f"{acc:.2f}%",
        "Macro F1-Score": f"{f1:.4f}",
        "Typo Queries Acc (%)": f"{typo_acc:.2f}% ({data['typo_correct']}/{data['typo_total']})",
        "Clean Queries Acc (%)": f"{clean_acc:.2f}% ({data['clean_correct']}/{data['clean_total']})",
        "Avg Latency (ms)": f"{avg_lat:.2f} ms",
        "P95 Latency (ms)": f"{p95_lat:.2f} ms"
    })

df_summary = pd.DataFrame(summary_data)
from IPython.display import display
print("=========================================================================================")
print("🏆 สรุปผลการทดสอบที่เที่ยงตรง (5-Fold Stratified Cross-Validation Results)")
print("=========================================================================================")
display(df_summary)


🏆 สรุปผลการทดสอบที่เที่ยงตรง (5-Fold Stratified Cross-Validation Results)


,Architecture Method,Fair Accuracy (%),Macro F1-Score,Typo Queries Acc (%),Clean Queries Acc (%),Avg Latency (ms),P95 Latency (ms)
0,1. Pure BERT,65.44%,0.7322,27.27% (3/11),68.80% (86/125),12.50 ms,14.37 ms
1,2. Edit Distance + BERT,77.94%,0.8101,54.55% (6/11),80.00% (100/125),20.83 ms,22.13 ms
2,3. Rules + Pure BERT,75.74%,0.7780,36.36% (4/11),79.20% (99/125),5.08 ms,14.23 ms
3,4a. 4-Tier Pipeline (No FewShot),77.94%,0.7842,54.55% (6/11),80.00% (100/125),9.59 ms,20.85 ms
4,4b. Full 4-Tier (Leak-Free 5-Fold CV),87.50%,0.8742,100.00% (11/11),86.40% (108/125),10.12 ms,22.19 ms


## 🔍 Step 6: วิเคราะห์การทำนายผิดพลาดเชิงลึก (Fair Misclassification Analysis)

In [6]:
df_logs = pd.DataFrame(detailed_logs)
pure_misses = df_logs[(df_logs["Method"] == "1. Pure BERT") & (~df_logs["Is_Correct"])]
cv_4tier_misses = df_logs[(df_logs["Method"] == "4b. Full 4-Tier (Leak-Free 5-Fold CV)") & (~df_logs["Is_Correct"])]

print(f"❌ จำนวนเคสที่ Pure BERT ทำนายผิดพลาด: {len(pure_misses)} เคส")
if not pure_misses.empty:
    display(pure_misses[["Fold", "Query", "Has_Typo", "Expected", "Predicted", "Latency_ms"]].head(10))

print(f"\n❌ จำนวนเคสที่ Full 4-Tier (Leak-Free 5-Fold CV) ทำนายผิดพลาด: {len(cv_4tier_misses)} เคส")
if not cv_4tier_misses.empty:
    display(cv_4tier_misses[["Fold", "Query", "Has_Typo", "Expected", "Predicted", "Latency_ms"]])


❌ จำนวนเคสที่ Pure BERT ทำนายผิดพลาด: 47 เคส


,Fold,Query,Has_Typo,Expected,Predicted,Latency_ms
5,1,หาเกงยีนส์ทรงหลวมสีดำ เอ็ม ใส่สบายๆ,True,product_search,size_recommendation,13.7338
10,1,เสื้อคอกมสีขาว ไซส์ L,True,product_search,size_recommendation,10.6503
15,1,ขอดูเสื้อ oversize ผ้า ultrasoft หน่อย,False,product_search,size_recommendation,13.5506
25,1,ขอดูกางเกงคาร์โก้สักตัว,False,product_search,size_recommendation,12.4485
30,1,มีกางเกงยีนส์สีน้ำเงิน ไซส์ 32 มั้ย,False,product_search,size_recommendation,10.8664
40,1,หาเสื้อสีดำใส่วิ่งออกกำลังกายได้ ไม่เกิน 350,False,product_search,size_recommendation,13.4013
50,1,มีกางเกงขาสั้นใส่ออกกำลังกายมั้ยครับ งบ 400,False,product_search,size_recommendation,13.3867
105,1,หาเสื้อยืดใส่ออกกำลังกาย แห้งเร็ว สีดำ s,False,product_search,size_recommendation,11.4903
120,1,ขอดูเสื้อยืดสีเขียวทุกรุ่นหน่อย,False,product_search,size_recommendation,11.3284
125,1,มีกางเกงยีนส์ลดราคาไหม,False,promotion_discount,product_search,13.6994



❌ จำนวนเคสที่ Full 4-Tier (Leak-Free 5-Fold CV) ทำนายผิดพลาด: 17 เคส


,Fold,Query,Has_Typo,Expected,Predicted,Latency_ms
89,1,ผ้า supima cotton ดีกว่าปกติยังไง,False,fabric_comparison,product_search,16.1561
129,1,มีกางเกงยีนส์ลดราคาไหม,False,promotion_discount,product_search,13.7226
144,2,มีเสื้อคอวีผ้านุ่มสีครีมมั้ย,False,product_search,fabric_comparison,2.7078
164,2,หาเสื้อยืดสีครีมผ้านุ่มไซส์ m,False,product_search,fabric_comparison,4.8151
254,2,หาเสื้อยืดสีน้ำเงินเข้ม ผ้านุ่ม ทรง unisex xl,False,product_search,fabric_comparison,9.6907
294,3,เสื้อแขนยาวผ้าไม่ยับมีไหมครับ,False,product_search,fabric_comparison,18.8427
329,3,ขอดูเสื้อยืดสีน้ำตาลอ่อนผ้านุ่มหน่อยค่ะ,False,product_search,fabric_comparison,7.1480
389,3,มีเสื้อ crop ผ้านุ่มๆ สีชมพูสวยๆ ไหมคะ,False,product_search,fabric_comparison,4.3690
394,3,มีกางเกงคาร์โก้ผ้านุ่มสีน้ำตาลมั้ยครับ,False,product_search,fabric_comparison,6.1667
409,3,ขอดูโปรประจำเดือนหน่อย,False,promotion_discount,product_search,0.1764
